# Export models to ONNX for the web demo

The web demo cannot use PyTorch, because Windows Smart App Control blocks torch's
unsigned DLLs. ONNX Runtime is signed by Microsoft and loads fine, so models are
converted once here and run locally through ONNX.

This exports **two** models:

1. **`cnn_damage_classifier.onnx`** — your trained CarDD damage-type model.
2. **`vehicle_check.onnx`** — stock ImageNet MobileNetV2, used to answer "is this
   even a vehicle, and what kind". Uses pretrained weights as-is, so there is
   nothing to train.

No retraining happens here. A CPU runtime is fine; the whole notebook takes a
couple of minutes.

**Requires** in `MyDrive/cardd/`: `cardd_src.zip` and
`results_mobilenet_v2/cnn_damage_classifier.pth`.

## 1. Mount Drive and install ONNX tooling

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# onnxscript is required by torch's current default (dynamo) ONNX exporter.
!pip install -q onnx onnxruntime onnxscript

## 2. Settings

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/cardd'
PROJECT   = '/content/AI-Image-Processing'
BACKBONE  = 'mobilenet_v2'          # must match the run that produced the checkpoint

CKPT = f'{DRIVE_DIR}/results_{BACKBONE}/cnn_damage_classifier.pth'

## 3. Unpack the repo source

In [ ]:
import os, zipfile
from pathlib import Path

assert Path(CKPT).exists(), f"Checkpoint not found: {CKPT}"

os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(f'{DRIVE_DIR}/cardd_src.zip') as zf:
    zf.extractall(PROJECT)
print("source unpacked")

## 4. Shared export helper

Two things every export here must guarantee:

- **A single self-contained file.** Torch's dynamo exporter writes weights to a
  separate `<name>.onnx.data` sidecar, which breaks the moment the `.onnx` travels
  without it. We collapse the weights back inline and assert the result is big
  enough to actually contain them.
- **A working exporter.** If the dynamo path is unhappy, fall back to the legacy
  TorchScript exporter, which handles a plain torchvision CNN perfectly well.

In [ ]:
import os
import onnx
import torch

IMG_SIZE = 224
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

opts = dict(
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)


def export_single_file(module, path, min_mb=5):
    module.eval()
    try:
        torch.onnx.export(module, dummy, path, **opts)
        how = 'dynamo'
    except Exception as e:
        print(f"  dynamo exporter failed ({type(e).__name__}), using legacy path")
        torch.onnx.export(module, dummy, path, dynamo=False, **opts)
        how = 'legacy'

    onnx.save_model(onnx.load(path), path, save_as_external_data=False)
    if os.path.exists(path + '.data'):
        os.remove(path + '.data')

    mb = os.path.getsize(path) / 1e6
    print(f"  {os.path.basename(path)}: {mb:.1f} MB (via {how})")
    assert mb > min_mb, "file too small - weights are still external, do not ship this"
    return path

## 5. Export the damage classifier

In [ ]:
import sys, json

sys.path.insert(0, f'{PROJECT}/src')
from model import build_model

ckpt        = torch.load(CKPT, map_location='cpu', weights_only=False)
class_names = ckpt['class_names']
img_size    = ckpt.get('img_size', IMG_SIZE)
backbone    = ckpt.get('backbone', BACKBONE)
assert img_size == IMG_SIZE, f"checkpoint img_size {img_size} != {IMG_SIZE}"

net = build_model(len(class_names), backbone=backbone, pretrained=False)
net.load_state_dict(ckpt['state_dict'])
net.eval()

print(f"backbone   : {backbone}")
print(f"class_names: {class_names}")

DAMAGE_ONNX = export_single_file(net, '/content/cnn_damage_classifier.onnx')

with open('/content/model_meta.json', 'w') as f:
    json.dump({
        'class_names': class_names,
        'img_size': img_size,
        'backbone': backbone,
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225],
    }, f, indent=2)
print("wrote model_meta.json")

## 6. Export the vehicle-check model

Stock ImageNet MobileNetV2. Its 1000 categories include both vehicles and the
things people accidentally upload instead (bicycles, motorbikes), so one model
answers "is this a vehicle" and "what kind" at once.

Same input size and same ImageNet normalisation as the damage model, so the web app
preprocesses the image **once** and feeds both sessions.

In [ ]:
from torchvision import models

weights = models.MobileNet_V2_Weights.DEFAULT
gate = models.mobilenet_v2(weights=weights)
categories = list(weights.meta['categories'])
print(f"{len(categories)} ImageNet categories")

GATE_ONNX = export_single_file(gate, '/content/vehicle_check.onnx')

### Which categories count as a vehicle

Resolved by name rather than by hard-coded index, so a mismatch shows up as a printed
warning here instead of silently mislabelling things at runtime. Check the resolved
list below looks sensible before moving on.

In [ ]:
VEHICLE_NAMES = [
    'sports car', 'convertible', 'cab', 'jeep', 'limousine', 'minivan', 'pickup',
    'beach wagon', 'racer', 'police van', 'ambulance', 'fire engine',
    'garbage truck', 'tow truck', 'trailer truck', 'moving van', 'school bus',
    'minibus', 'recreational vehicle', 'Model T', 'golfcart', 'go-kart',
    'snowplow', 'trolleybus', 'half track', 'tractor',
]

# Vehicle PARTS are tracked separately. The damage model is trained on cropped
# damage regions, so a valid upload frequently shows a wheel or a grille rather
# than a whole car - without these, genuine damage photos get flagged as
# "may not be a vehicle". They count as evidence, but never as the reported type:
# "car wheel" is not an answer to "what type of car is this".
VEHICLE_PART_NAMES = ['car wheel', 'car mirror', 'disk brake', 'grille']

index_of = {name: i for i, name in enumerate(categories)}
missing = [n for n in VEHICLE_NAMES + VEHICLE_PART_NAMES if n not in index_of]
resolved = {n: index_of[n] for n in VEHICLE_NAMES if n in index_of}
parts = sorted(index_of[n] for n in VEHICLE_PART_NAMES if n in index_of)

if missing:
    print("WARNING - these names are not in this torchvision version's category list:")
    for n in missing:
        print(f"    {n!r}")
    print("  They will simply be skipped. Tell Claude if the list looks too short.\n")

print(f"{len(resolved)} whole-vehicle categories resolved:")
for name, i in sorted(resolved.items(), key=lambda kv: kv[1]):
    print(f"  {i:4d}  {name}")
print(f"\n{len(parts)} vehicle-part categories (evidence only):")
for i in parts:
    print(f"  {i:4d}  {categories[i]}")

assert len(resolved) >= 15, "too few vehicle categories matched - do not ship this"
assert len(parts) >= 3, "vehicle parts missing - damage crops will be misflagged"

with open('/content/vehicle_check_meta.json', 'w') as f:
    json.dump({
        'categories': categories,
        'vehicle_indices': sorted(resolved.values()),
        'vehicle_part_indices': parts,
        'img_size': IMG_SIZE,
        'source': 'torchvision MobileNet_V2_Weights.DEFAULT (ImageNet-1k)',
    }, f, indent=2)
print("\nwrote vehicle_check_meta.json")

## 7. Verify the damage export matches PyTorch

Same random inputs through both runtimes. Max absolute difference should be about
`1e-5` or smaller — float round-off, not a behaviour change. If this prints
something large, do **not** ship the ONNX file.

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(DAMAGE_ONNX, providers=['CPUExecutionProvider'])

worst = 0.0
for _ in range(5):
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        expected = net(x).numpy()
    actual = sess.run(None, {'input': x.numpy()})[0]
    worst = max(worst, float(np.abs(expected - actual).max()))

print(f"max abs difference over 5 random inputs: {worst:.3e}")
print("PASS - export is faithful" if worst < 1e-4 else "FAIL - do not use this file")

## 8. Sanity-check both models on a real image

Runs one actual test image through the damage classifier and the vehicle check.
Change `TEST_CLASS` to try different damage types.

Expect the vehicle score to be **low on tight damage crops** — those do not look like
a whole car to ImageNet. That is exactly why the web app treats this as an advisory
warning rather than a hard gate.

In [ ]:
from PIL import Image

TEST_CLASS = 'tire_flat'   # crack | dent | glass_shatter | lamp_broken | scratch | tire_flat

folder = Path(f'{PROJECT}/data/classification/test/{TEST_CLASS}')
if not folder.exists():
    print("Test images are not unpacked in this session - skipping this check.")
else:
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    img = Image.open(sorted(folder.glob('*.jpg'))[0]).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    a = np.asarray(img, dtype=np.float32) / 255.0
    a = ((a - mean) / std).transpose(2, 0, 1)[None].astype(np.float32)

    def softmax(v):
        e = np.exp(v - v.max()); return e / e.sum()

    p = softmax(sess.run(None, {'input': a})[0][0])
    print(f"damage prediction (true class: {TEST_CLASS})")
    for i in np.argsort(p)[::-1][:3]:
        print(f"  {class_names[i]:15s} {p[i]*100:5.1f}%")

    gsess = ort.InferenceSession(GATE_ONNX, providers=['CPUExecutionProvider'])
    g = softmax(gsess.run(None, {'input': a})[0][0])
    vehicle_score = float(g[sorted(resolved.values())].sum())
    print(f"\nvehicle score: {vehicle_score*100:.1f}%")
    print("top ImageNet guesses:")
    for i in np.argsort(g)[::-1][:3]:
        print(f"  {categories[i]:25s} {g[i]*100:5.1f}%")

## 9. Copy everything to Drive

In [ ]:
import shutil

dest = Path(DRIVE_DIR) / 'onnx'
dest.mkdir(parents=True, exist_ok=True)

for f in ('/content/cnn_damage_classifier.onnx', '/content/model_meta.json',
          '/content/vehicle_check.onnx', '/content/vehicle_check_meta.json'):
    shutil.copy2(f, dest / Path(f).name)
    print(f"saved {Path(f).name:28s} {os.path.getsize(f)/1e6:6.1f} MB")

print(f"\nDownload all four from Drive at: {dest}")

## Next: update the web demo

The demo lives in a **separate repo** (`car-damage-demo`), deployed on Vercel, so this
training repo stays free of deployment machinery.

1. Download all four files from `MyDrive/cardd/onnx/`.
2. Copy them into `car-damage-demo/models/`, replacing what is there.
3. Run it locally to check:

```
cd ..\car-damage-demo
.venv\Scripts\python webapp\app.py      # then open http://127.0.0.1:5000
```

4. Ship it. Vercel redeploys automatically on push:

```
git add models && git commit -m "Update models" && git push
```

Both models are exported as **single self-contained files** — cell 4 asserts this. An
`.onnx` that leaves its `.onnx.data` sidecar behind loads fine locally and then fails
on the server.